# Week 12 guided practice

CML3112. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv into Colab with the Files panel before running the cells. The target in this notebook is cooling load in kW.

## Today’s learning

*PowerPoint slide 2*

WHAT WE WILL DO
Compare candidate models on the same rows, apply the limits an engineer sets, and explain the choice. Meet random forests and boosting, and write the limits of a model in plain words.

HOW YOU WILL SHOW IT
You can select a model that meets stated limits, say who is affected when it is wrong, and describe its purpose, data, scores and limits on one page.

## The supplied file

*PowerPoint slide 4*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep them beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.

## Loading the supplied observations

*PowerPoint slide 5*

raw_data: original CSV table; keep it unchanged so the analysis can be checked.

Run the cell and read the output below it.

In [ ]:
import pandas as pd
raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)

## Preparing the supplied modelling rows: step 1

*PowerPoint slide 6*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))

## Choose groups before choosing rows

*PowerPoint slide 7*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list.

Here E1–E4 supply training rows, E5 validation rows and E6 test rows.

The groups stay separate so one equipment group cannot appear on both sides.

## Preparing the supplied modelling rows: step 2

*PowerPoint slide 8*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 9*

training_data: rows used to learn model settings; only these rows may determine imputation values.

validation_data: separate rows used to compare models without fitting on them.

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 10*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))

## The Week 8 line, rebuilt in a few lines

*PowerPoint slide 11*

Every line repeats Week 8: split into X and y, fit the pipeline, predict, score.
The names X_valid, linear_model and mae are used again in the next cells.
The printed number is the validation MAE of the line.

The same rows, the same pipeline, the same MAE as Week 8.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
linear_model = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression())
linear_model.fit(X_train, y_train)
validation_predictions = linear_model.predict(X_valid)
mae = mean_absolute_error(
    validation_data["target"], validation_predictions)
print(round(mae, 4))

**Example output**

```text
0.7427
```

## Three candidates on the same validation rows

*PowerPoint slide 12*

All three were scored on the same rows, in the same unit.
Speed comes next: a candidate must also be fast enough.

## Candidate evidence and latency

*PowerPoint slide 13*

The CSV supplies observations. We build the table of candidates and timings ourselves. Calculate validation MAE from its held-out observations.

Measure prediction time on this computer. Timing varies by machine and repeated run, so record the measurement conditions.

## Operations for the next example

*PowerPoint slide 14*

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.

## Measuring elapsed time

*PowerPoint slide 15*

perf_counter() reads a high-resolution clock in seconds. Subtract the earlier reading from the later one, then multiply by 1000 for milliseconds. One measured run is an illustration and varies from run to run.

## Time the prediction step

*PowerPoint slide 16*

perf_counter reads seconds from a high-resolution clock.

Read it immediately before and after predict.

Convert elapsed seconds to milliseconds and divide by row count.

One timing is illustrative: repeated measurements are needed for a reliable speed claim.

## A measured candidate from the supplied validation data: step 1

*PowerPoint slide 17*

prediction_start_time: timer reading immediately before predicting the batch.

Run the cell and read the output below it.

In [ ]:
from time import perf_counter
prediction_start_time = perf_counter()
linear_model.predict(X_valid)
prediction_time_ms = (
    (perf_counter() - prediction_start_time)
    * 1000
    / len(validation_data)
)

## A measured candidate from the supplied validation data: step 2

*PowerPoint slide 18*

candidate_models: table of model names, measured validation errors and prediction times.

## A measured candidate from the supplied validation data: step 2

*PowerPoint slide 19*

candidate_models: one row per model, with its name, validation error and prediction time.

Run the cell and read the output below it.

In [ ]:
candidate_models = pd.DataFrame(
    {
        "name": ["linear"],
        "validation_mae": [float(mae)],
        "latency_ms": [prediction_time_ms],
    }
)
print(candidate_models)

## The same recipe gives one row per candidate

*PowerPoint slide 20*

Same recipe for every row: fit, predict, score the MAE, time the prediction.
Latency depends on your computer, so each student measures it.

## Constraints and tie-breaks

*PowerPoint slide 21*

For this classroom decision, use MAE at most 10 target units and mean latency at most 10 ms per observation. These are stated practice requirements.

Filter both limits inclusively. sort_values ranks eligible rows by validation_mae, then latency_ms, then name. empty detects no eligible rows.

## Operations for the next example

*PowerPoint slide 22*

01

Checking for no rows: equipment_data.empty is True when the filtered table has no entries. Check it before attempting to select a first row. empty is a property, without parentheses.

02

Sorting by several columns: equipment_data.sort_values(["validation_mae", "latency_ms", "name"]) sorts ascending by error first. Equal errors use latency, then alphabetical name as tie-breaks.

03

Selecting by row position: equipment_data.iloc[0] selects the first row by its position. ["name"] then selects that row’s name field. An empty table has no row at position zero.

## Checking a condition before calculating

*PowerPoint slide 23*

if condition: runs its indented statements only when the condition is True. else: handles the other case. Check for a nonzero denominator before division, and for a nonempty table before selecting its first row.

## Selection from the measured candidate table: step 1

*PowerPoint slide 24*

eligible_models: candidates satisfying both stated limits.

Run the cell and read the output below it.

In [ ]:
error_is_acceptable = candidate_models["validation_mae"] <= 10
time_is_acceptable = candidate_models["latency_ms"] <= 10
model_is_eligible = error_is_acceptable & time_is_acceptable

## Apply both model-selection rules

*PowerPoint slide 25*

A candidate must satisfy both the error limit and the time limit.

Compare each column with its limit first, then combine the True/False answers with &.

Filtering can return no rows.

## Selection from the measured candidate table: step 2

*PowerPoint slide 26*

eligible_models: candidates satisfying both stated limits.

Run the cell and read the output below it.

In [ ]:
eligible_models = candidate_models[model_is_eligible]

## Choose only if an eligible model exists

*PowerPoint slide 27*

sort_values orders by error, then time, then name. empty answers whether no rows remain.

Check it before iloc[0], which requests the first row by position.

Return None when no candidate qualifies.

## Selection from the measured candidate table: step 3

*PowerPoint slide 28*

eligible_models: candidates satisfying both stated limits.

Run the cell and read the output below it.

In [ ]:
ranked_models = eligible_models.sort_values(
    ["validation_mae", "latency_ms", "name"]
)
if ranked_models.empty:
    selected_model_name = None
else:
    selected_model_name = ranked_models.iloc[0]["name"]
print(selected_model_name)

## Interpreting the recommendation

*PowerPoint slide 29*

State the data, split, metric, timing method and constraints. Report when the measured model fails a requirement.

Tune hyperparameters using validation or cross-validation. Preserve the final test until the choice is fixed.

## A random forest averages many trees

*PowerPoint slide 30*

Illustrative numbers. Each tree sees different rows, and the forest averages their answers.

## A forest uses the same create, fit, predict pattern

*PowerPoint slide 31*

RandomForestRegressor(
    n_estimators=100,
    random_state=17)

n_estimators is the number of trees. random_state fixes the random choices so everyone gets the same forest.
Use it exactly like the other models: forest.fit(X_train, y_train), then forest.predict(X_valid).
Score it with the same MAE on the same validation rows, and compare it with the baseline and the line.

## Boosting adds small trees that fix what is still wrong

*PowerPoint slide 32*

Illustrative numbers. Named here and taught in Semester 2.

## Who is affected when a model is wrong

*PowerPoint slide 34*

A MISSED ALERT (FALSE NEGATIVE)
Suppose an alert fires when the predicted cooling load passes a limit we choose. If a real problem goes unwarned, equipment can be damaged and people nearby can be put at risk.
Check that the training data looks like where the model will run. A new machine may behave differently from E1 to E4.

A FALSE ALARM (FALSE POSITIVE)
Inspections are wasted, and operators learn to ignore the alerts. A model nobody trusts protects nobody.
Choose the threshold with the people who live with the consequences.

## Take care with data about people

*PowerPoint slide 35*

MACHINE DATA
Sensor readings from a motor, a building plant or a crusher carry little personal risk.
Data about workers, drivers or customers is different. It can identify a person.

WHAT TO DO
Uganda's Data Protection and Privacy Act, 2019 protects personal data. Collect only what you need, keep it safe, and say what it is used for.
This is a summary for learning and is not legal advice.

## Your issued task

*PowerPoint slide 36*

01

Investigate building cooling load.

02

Target units: kW.

03

These are synthetic teaching records for practice.

04

Each record is one operating observation; equipment_id repeats across observations.

## select_model(candidates, max_mae, max_ms)

*PowerPoint slide 37*

Candidates have name, validation_mae and latency_ms.

Keep those meeting both inclusive constraints; select smallest validation_mae, then latency_ms, then name alphabetically.

Return the selected name or None.

Never use test scores for selection.

Keep the exact names and outputs. Check types, units and the issued brief.

## regression_errors(actual, predicted)

*PowerPoint slide 38*

Return mae and rmse, calculated over the same paired observations, with full precision.

Keep the exact names and outputs. Check types, units and the issued brief.

## Written evidence

*PowerPoint slide 39*

01

Use the cell headed “My written evidence”.

02

Explain the supplied scenario in your own words.

03

State target units, reasons and limitations.

04

Submit one complete notebook containing code and written evidence.